# EthicAgent — Quick Start Guide

**A Context-Aware Neuro-Symbolic Framework for Ethical Autonomous Decision-Making in Agentic AI Systems**

This notebook provides a quick introduction to EthicAgent, demonstrating:
1. Installation & setup
2. Basic ethical evaluation
3. Understanding the EDS formula
4. Interpreting results

---

## 1. Setup & Imports

First, install the required dependencies and import the core modules.

In [ ]:
import sys
sys.path.insert(0, '..')

# Core imports
from ethicagent.core.state import PipelineState, PipelineStage
from ethicagent.core.logger import AuditLogger
from ethicagent.ethics.ethical_score import EthicalDecision, EthicalVerdict, PhilosophyResult
from ethicagent.agents.ethical_reasoner import EthicalReasonerAgent
from ethicagent.agents.context_agent import ContextAgent
from ethicagent.utils.helpers import format_score_breakdown

print('EthicAgent modules loaded successfully!')

## 2. The EDS Formula

The core innovation of EthicAgent is the **Ethical Decision Score (EDS)**:

$$EDS(a) = w_1 \cdot D(a) + w_2 \cdot C(a) + w_3 \cdot V(a) + w_4 \cdot Ctx(a)$$

Where:
- $D(a)$ — Deontological score (rule-based ethical assessment)
- $C(a)$ — Consequentialist score (outcome-based assessment)
- $V(a)$ — Virtue Ethics score (character/fairness assessment)
- $Ctx(a)$ — Contextual Ethics score (situational assessment)
- $w_1, w_2, w_3, w_4$ — Domain-specific weights ($\sum w_i = 1$)

### Decision Thresholds
| EDS Range | Verdict | Description |
|-----------|---------|-------------|
| $\geq 0.80$ | AUTO_APPROVE | Action is ethically sound |
| $0.50 \leq EDS < 0.80$ | ESCALATE | Requires human review |
| $< 0.50$ | REJECT | Action is ethically problematic |
| $D(a) = 0$ | HARD_BLOCK | Deontological violation — blocked regardless |

## 3. Basic Ethical Evaluation

Let's compute the EDS for a simple healthcare scenario.

In [ ]:
# Initialize the ethical reasoner
reasoner = EthicalReasonerAgent()

# Define philosophy scores for a healthcare triage scenario
scores = {
    'deontological': 0.85,    # D(a): respects patient rights
    'consequentialist': 0.70,  # C(a): outcomes benefit more patients
    'virtue_ethics': 0.80,     # V(a): demonstrates fairness and care
    'contextual': 0.75,        # Ctx(a): appropriate for emergency context
}

# Healthcare domain weights
healthcare_weights = {
    'deontological': 0.35,
    'consequentialist': 0.25,
    'virtue_ethics': 0.20,
    'contextual': 0.20,
}

# Compute EDS
eds = reasoner.compute_eds(scores, healthcare_weights)
print(f'EDS = {eds:.4f}')
print(f'\nBreakdown:')
for phil, score in scores.items():
    w = healthcare_weights[phil]
    contribution = w * score
    print(f'  {phil:20s}: {w:.2f} × {score:.2f} = {contribution:.4f}')
print(f'  {"":20s}  Total EDS = {eds:.4f}')

In [ ]:
# Determine verdict
verdict = reasoner.determine_verdict(scores, healthcare_weights)
print(f'Verdict: {verdict.upper()}')
print(f'EDS = {eds:.4f} → {"AUTO_APPROVE" if eds >= 0.8 else "ESCALATE" if eds >= 0.5 else "REJECT"}')

## 4. Domain-Specific Weights

Different domains prioritize different ethical philosophies. Let's compare how the same scores produce different EDS values across domains.

In [ ]:
# Same scores, different domain weights
test_scores = {
    'deontological': 0.75,
    'consequentialist': 0.85,
    'virtue_ethics': 0.60,
    'contextual': 0.90,
}

domain_weights = {
    'Healthcare': {'deontological': 0.35, 'consequentialist': 0.25, 'virtue_ethics': 0.20, 'contextual': 0.20},
    'Finance':    {'deontological': 0.20, 'consequentialist': 0.25, 'virtue_ethics': 0.35, 'contextual': 0.20},
    'Hiring':     {'deontological': 0.15, 'consequentialist': 0.20, 'virtue_ethics': 0.40, 'contextual': 0.25},
    'Disaster':   {'deontological': 0.20, 'consequentialist': 0.35, 'virtue_ethics': 0.15, 'contextual': 0.30},
    'General':    {'deontological': 0.25, 'consequentialist': 0.25, 'virtue_ethics': 0.25, 'contextual': 0.25},
}

print(f'{"Domain":12s} | {"EDS":>8s} | {"Verdict":>12s}')
print('-' * 40)
for domain, weights in domain_weights.items():
    eds = reasoner.compute_eds(test_scores, weights)
    verdict = reasoner.determine_verdict(test_scores, weights)
    print(f'{domain:12s} | {eds:8.4f} | {verdict:>12s}')

## 5. Hard Block Override

When the deontological score is zero (a fundamental ethical rule is violated), the system issues a **HARD_BLOCK** regardless of other scores.

In [ ]:
# Hard block example: deontological violation
violation_scores = {
    'deontological': 0.0,     # Rule violation!
    'consequentialist': 0.95,  # Great outcomes
    'virtue_ethics': 0.90,     # Looks fair
    'contextual': 0.88,        # Appropriate context
}

equal_weights = {
    'deontological': 0.25,
    'consequentialist': 0.25,
    'virtue_ethics': 0.25,
    'contextual': 0.25,
}

eds = reasoner.compute_eds(violation_scores, equal_weights)
verdict = reasoner.determine_verdict(violation_scores, equal_weights)

print(f'EDS = {eds:.4f} (would normally be APPROVE)')
print(f'Verdict: {verdict.upper()} — Deontological override active!')
print(f'\nKey insight: Even with EDS = {eds:.4f}, D(a) = 0 triggers HARD_BLOCK.')
print('This ensures fundamental ethical rules can never be bypassed.')

## 6. Pipeline State Management

EthicAgent tracks the full decision pipeline through stages.

In [ ]:
# Create and inspect pipeline state
state = PipelineState(
    task='Allocate ventilator to Patient A over Patient B',
    domain='healthcare',
    context={'urgency': 'critical', 'patients': 2, 'resources': 1}
)

print(f'Task: {state.task}')
print(f'Domain: {state.domain}')
print(f'Stage: {state.stage}')
print(f'Context: {state.context}')

# Show all pipeline stages
print(f'\nPipeline Stages:')
for stage in PipelineStage:
    print(f'  {stage.value}')

## 7. Audit Logging

Every decision is logged for transparency and accountability.

In [ ]:
# Create audit logger
logger = AuditLogger()

# Log some events
logger.log_event('PIPELINE_START', {'task': 'Ventilator allocation', 'domain': 'healthcare'})
logger.log_event('CONTEXT_ANALYSIS', {'domain_detected': 'healthcare', 'urgency': 'critical'})
logger.log_decision(
    task='Allocate ventilator to Patient A',
    verdict='escalate',
    eds_score=0.72,
    scores={'deontological': 0.75, 'consequentialist': 0.70, 'virtue_ethics': 0.68, 'contextual': 0.74}
)

# Display audit trail
print(f'Audit entries: {len(logger.entries)}')
for entry in logger.entries:
    print(f'  [{entry.timestamp[:19]}] {entry.event_type}: {entry.data}')

## 8. Context Analysis

The Context Agent automatically detects domains, extracts entities, and assesses urgency.

In [ ]:
# Initialize context agent
context_agent = ContextAgent()

# Analyze different tasks
tasks = [
    'Approve a $500,000 mortgage application for a first-time homebuyer',
    'Perform emergency triage for mass casualty event',
    'Screen job applicants for senior software engineer position',
    'Deploy search and rescue teams after earthquake',
]

print(f'{"Task":60s} | {"Domain":12s} | {"Urgency"}')
print('-' * 95)
for task in tasks:
    domain = context_agent.classify_domain(task)
    urgency = context_agent.determine_urgency(task)
    print(f'{task:60s} | {domain:12s} | {urgency}')

## 9. Ethical Decision Object

The `EthicalDecision` object encapsulates all aspects of an ethical evaluation.

In [ ]:
# Create a complete EthicalDecision
decision = EthicalDecision(
    task='Approve loan for applicant with borderline credit score',
    domain='finance',
    verdict=EthicalVerdict.ESCALATE,
    eds_score=0.68,
    philosophy_results=[
        PhilosophyResult('deontological', 0.75, 'Follows fair lending rules', ['Compliant with ECOA']),
        PhilosophyResult('consequentialist', 0.60, 'Mixed outcomes expected', ['Default risk moderate']),
        PhilosophyResult('virtue_ethics', 0.65, 'Fair but borderline', ['Income ratio concerning']),
        PhilosophyResult('contextual', 0.72, 'Local market supports', ['First-time buyer program']),
    ],
    reasoning='Borderline case requires human review due to mixed signals across philosophies.',
    conflicts=[],
    confidence=0.65,
)

# Display summary
print(decision.summary())
print(f'\nDict keys: {list(decision.to_dict().keys())}')

## Next Steps

- **[02_scenario_analysis.ipynb](02_scenario_analysis.ipynb)** — Run and analyze built-in scenarios
- **[03_evaluation_results.ipynb](03_evaluation_results.ipynb)** — Benchmark results and statistical analysis
- **[04_custom_scenarios.ipynb](04_custom_scenarios.ipynb)** — Create your own custom scenarios

---
*EthicAgent: Context-Aware Neuro-Symbolic Framework for Ethical Autonomous Decision-Making*